# 数据结构与科学计算 · 第 01 讲
## 数组、复杂度与二维 CFD 网格

> **课程定位**：从 Python 基础出发，补上计算机科学中的数据结构知识，并逐渐连接 CFD、AI4PDE、JAX 和强化学习。你不需要先学过算法课，也不需要独立写过求解器。

**本讲预计用时**：60–90 分钟（包括动手练习）  
**运行环境**：Python 3.10+、NumPy、Jupyter Notebook（推荐 VS Code）  
**这一讲的学习目标**：

1. 区分**数据结构**和**算法**，理解为什么选择数据结构会改变程序性能。
2. 能读懂常见的时间复杂度：\(O(1)\)、\(O(n)\)、\(O(n^2)\)。
3. 会用 Python `list`、二维嵌套列表与 NumPy `ndarray`，知道它们的根本区别。
4. 理解**索引、存储顺序、切片视图、复制与别名**，避开数值计算的经典错误。
5. 用一个二维标量场的离散 Laplace 算子，把数据结构知识连接到 CFD。

**建议使用方法**：按顺序运行每个代码单元；遇到“先预测”时先想再运行；最后独立做练习，再看参考答案。

## 课程路线图

| 讲次 | 主题 | Python 工具 | 科学计算中的联系 |
|:--|:--|:--|:--|
| **01（当前）** | 数组、复杂度、内存布局 | `list`、`tuple`、`numpy.ndarray` | 二维网格、场变量、离散算子 |
| 02 | 栈、队列、双端队列 | `list`、`collections.deque` | 时间步任务队列、搜索顺序 |
| 03 | 哈希表与集合 | `dict`、`set` | 网格编号、去重、状态缓存 |
| 04 | 链表、动态数组与内存 | 手写节点、`list` | 动态扩容、内存与缓存局部性 |
| 05 | 树、堆与优先队列 | `heapq` | 自适应网格、事件优先级 |
| 06 | 图及其遍历 | 邻接表、BFS、DFS | 非结构网格、单元邻接 |
| 07 | 稀疏矩阵 | COO、CSR、SciPy | 压力 Poisson 方程与线性系统 |
| 08 | 工程实践与 JAX 连接 | profiling、vectorization、PyTree | 数组布局、GPU、可微计算 |

**贯穿问题**：一个 CFD 程序里，速度场、网格拓扑、稀疏算子、状态缓存应该分别用什么结构保存？

## 0. 环境准备：在 VS Code 里运行

把这个 `.ipynb` 文件放进一个文件夹，在 VS Code 打开。安装官方的 **Python** 与 **Jupyter** 扩展，然后在右上角 **Select Kernel / 选择内核** 中选择你的 Python 环境。

如果你习惯使用 `uv`，在该文件夹的终端运行（新项目可以直接这样建立环境）：

```bash
uv venv
uv pip install numpy ipykernel jupyter
```

Windows 与 macOS 都可以使用以上命令；确保 VS Code 选中该文件夹下的 `.venv` 作为 Python 环境。然后点击每个代码单元左侧的 **▶**，或使用 `Shift+Enter` 顺序执行。

**Notebook 操作小抄**：
- **Code 单元**：真正执行 Python；它保存的变量可以被后面的单元使用。
- **Markdown 单元**：写公式、解释与学习笔记；支持 LaTeX，例如 `$O(n)$`。
- **Run All / 运行全部**：从头运行，检查是否有隐藏状态依赖。
- **Restart Kernel / 重启内核**：清空内存里的变量。发生奇怪错误时很有用。

运行下面的单元，确认你的环境：

In [1]:
import sys
import numpy as np

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("环境检查通过 ✅")

Python: 3.13.5
NumPy: 2.3.5
环境检查通过 ✅


---
## 1. 什么是数据结构？

**数据结构（Data Structure）**：组织、存储数据的方式，以及支持的操作。  
**算法（Algorithm）**：为解决问题而执行的一系列步骤。

例如在二维 CFD 中，给定一个 \(N_x\) 列、\(N_y\) 行的网格，保存速度 \(u(x,y)\) 可以有多种方式：

- 一层 Python 列表：`u_flat[k]`；
- 嵌套 Python 列表：`u[j][i]`；
- NumPy 二维数组：`u[j, i]`；
- 对稀疏网格或非结构网格，可能还需要索引表、邻接表等结构。

虽然它们都能表达“某个位置的数值”，但**元素类型、内存布局、索引操作、复制行为、计算速度**可能非常不同。

本课先围绕最普遍的数据结构——**数组（array）** 展开。注意：Python 的 `list` 不是等同于 NumPy 的数值数组。

## 2. 时间复杂度：为什么不只看代码行数？

当输入规模为 \(n\) 时，我们关心一个操作的运行成本随 \(n\) 如何变化。

| 复杂度 | 直观含义 | 例子 |
|:--|:--|:--|
| \(O(1)\) | 规模变大，单次操作工作量基本不变 | 列表按下标取元素 `a[i]` |
| \(O(\log n)\) | 规模翻倍，仅增加少量步骤 | 已排序序列的二分查找 |
| \(O(n)\) | 需要看过约 \(n\) 个元素 | 在线性列表查找某个值 |
| \(O(n\log n)\) | 常见高效排序的量级 | 比较排序 |
| \(O(n^2)\) | 输入规模翻倍，工作量约增至 4 倍 | 双重循环扫描 \(n\times n\) 对元素 |

> **重要区别**：`O(1)` 不等于耗时一定很短；大 O 只描述规模增长趋势，不直接代表实际秒数。常数、缓存、解释器开销、硬件也会影响速度。

若规则网格为 \(N_x\times N_y\)，遍历所有单元的工作量为

\[
T(N_x,N_y)=O(N_xN_y).
\]

若令 \(N_x=N_y=n\)，则写作 \(O(n^2)\)。这里的 \(n\) 指**单边格点数**，不是总格点数；分析时要先定义清楚变量。

In [2]:
# 观察 O(1) 索引与 O(n) 扫描在“所做工作”上的不同。
data = [4, 8, 15, 16, 23, 42]

print("按下标读取（不必扫描前面的元素）：", data[4])

comparisons = 0
target = 42
for value in data:
    comparisons += 1
    if value == target:
        break
print(f"线性查找 {target}：进行了 {comparisons} 次比较")

按下标读取（不必扫描前面的元素）： 23
线性查找 42：进行了 6 次比较


### 小思考（先别运行下一单元）

下面两段代码分别是什么时间复杂度？其中 \(n=\text{len}(a)\)。

```python
# A
x = a[0]

# B
s = 0
for x in a:
    s += x
```

<details>
<summary>展开答案</summary>

A 是 \(O(1)\)；B 是 \(O(n)\)。如果 `a` 为空，A 会抛出 `IndexError`，这与复杂度分析是两个问题。

</details>

## 3. Python 的 `list`：动态数组，不是数学向量

Python `list` 适合保存**一列对象的引用**，可以动态增加、删除、修改元素：

```python
a = [1, 2, 3]
a.append(4)     # 末尾加入
a[1] = 99       # 按下标替换
a.pop()         # 删除末尾元素
```

在 CPython 中，列表通常通过一个可扩容的、连续的**对象引用数组**实现。列表槽位放的是对 Python 对象的引用，不是把所有数字作为紧凑的双精度浮点数直接存入列表槽位。

因此：
- `a[i]`：按索引访问，\(O(1)\)；
- `a.append(x)`：**均摊** \(O(1)\)，偶尔扩容需要复制引用；
- `a.pop()`：末尾弹出，\(O(1)\)；
- `a.insert(0, x)`、`a.pop(0)`：通常 \(O(n)\)，需要移动元素引用；
- `x in a`：一般需要逐个比较，\(O(n)\)。

“均摊”意味着多次操作的**平均每次成本**，并不保证每次 `append` 都发生严格常数步骤。

In [3]:
a = [10, 20, 30]
a.append(40)
print("append 后:", a)

print("下标访问 a[2]:", a[2])
print("负下标 a[-1]:", a[-1])
print("切片 a[1:3]:", a[1:3])

a.insert(0, 5)
print("表头插入后:", a)

removed = a.pop()
print("末尾弹出:", removed, "剩余:", a)

append 后: [10, 20, 30, 40]
下标访问 a[2]: 30
负下标 a[-1]: 40
切片 a[1:3]: [20, 30]
表头插入后: [5, 10, 20, 30, 40]
末尾弹出: 40 剩余: [5, 10, 20, 30]


### 用小实验观察增长趋势

我们比较两种“往开头加入元素”的方法：

- `list.insert(0, x)`：后面的引用通常整体后移；
- `collections.deque.appendleft(x)`：双端队列专为两端操作优化。

下面记录的是**本机实测耗时**，不是严谨的性能基准；受运行状态影响，可能有波动。只观察随着规模增加，耗时如何变化。

In [4]:
from collections import deque
from time import perf_counter


def prepend_with_list(n: int) -> None:
    a = []
    for i in range(n):
        a.insert(0, i)


def prepend_with_deque(n: int) -> None:
    q = deque()
    for i in range(n):
        q.appendleft(i)


def elapsed(func, n: int) -> float:
    t0 = perf_counter()
    func(n)
    return (perf_counter() - t0) * 1000  # ms

for n in (1_000, 2_000, 4_000):
    t_list = min(elapsed(prepend_with_list, n) for _ in range(3))
    t_deque = min(elapsed(prepend_with_deque, n) for _ in range(3))
    print(f"n={n:5d} | list: {t_list:8.3f} ms | deque: {t_deque:8.3f} ms")

n= 1000 | list:    0.119 ms | deque:    0.022 ms
n= 2000 | list:    0.421 ms | deque:    0.044 ms
n= 4000 | list:    1.633 ms | deque:    0.090 ms


> `deque` 将是第 02 讲的主角。这里先形成一个习惯：**不是所有能完成任务的数据结构都同样适合这个任务**。同一种结构的不同操作，也可能有完全不同的复杂度。

### `tuple`：不可变序列

`tuple` 和 `list` 一样可以按索引访问，但它**不能修改已有元素**。例如 `grid_shape = (ny, nx)` 就很适合用元组表达“形状”这个固定结构。

In [5]:
grid_shape = (6, 8)
ny, nx = grid_shape
print("网格形状:", grid_shape)
print("ny=", ny, "nx=", nx)

# 下面这句如果取消注释，会抛出 TypeError：
# grid_shape[0] = 10

网格形状: (6, 8)
ny= 6 nx= 8


## 4. 经典错误：二维 Python 列表的“别名”

你可能会想用下面的方法创建一个 \(3\times4\) 的二维列表：

```python
grid = [[0] * 4] * 3
```

但是 `* 3` 复制的是**同一个内层列表的引用**。结果：修改一行，会让看起来不同的行都变动。

这个问题很容易出现在 CFD 网格、动态规划表和状态矩阵的代码中。

In [6]:
bad_grid = [[0] * 4] * 3
bad_grid[0][1] = 7
print("错误写法:", bad_grid)
print("前三行是否引用同一个对象:", bad_grid[0] is bad_grid[1])

# 每行独立构造一个新列表：
good_grid = [[0] * 4 for _ in range(3)]
good_grid[0][1] = 7
print("正确写法:", good_grid)
print("前三行是否引用同一个对象:", good_grid[0] is good_grid[1])

错误写法: [[0, 7, 0, 0], [0, 7, 0, 0], [0, 7, 0, 0]]
前三行是否引用同一个对象: True
正确写法: [[0, 7, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]
前三行是否引用同一个对象: False


## 5. NumPy `ndarray`：科学计算的基本数据容器

对 CFD 和机器学习，通常更适合把同类型的数值放在 `numpy.ndarray` 中。

理解它的四个属性：

- `shape`：每个维度的大小，例如 `(ny, nx)`；
- `dtype`：数值类型，例如 `float32`、`float64`；
- `ndim`：维度数量；
- `strides`：在当前内存视图中，某个维度的索引增加 1，要跨越多少**字节**。

NumPy 数组通常把同一类型的数值放在连续、紧凑的内存块中；但**切片、转置等数组视图不一定内存连续**。这是后面理解缓存、矢量化与 GPU 的基础。

In [7]:
ny, nx = 4, 6
u = np.zeros((ny, nx), dtype=np.float64)
u[1, 2] = 3.14

print(u)
print("shape:", u.shape)
print("ndim:", u.ndim)
print("dtype:", u.dtype)
print("strides (bytes):", u.strides)
print("nbytes:", u.nbytes)
print("是否 C 连续:", u.flags.c_contiguous)

[[0.   0.   0.   0.   0.   0.  ]
 [0.   0.   3.14 0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.  ]]
shape: (4, 6)
ndim: 2
dtype: float64
strides (bytes): (48, 8)
nbytes: 192
是否 C 连续: True


### 在二维网格里，`u[j, i]` 为什么这么写？

约定 \(i\) 表示 x 方向（列），\(j\) 表示 y 方向（行）。Python/NumPy 的二维索引通常写作

\[
u_{j,i}\quad\longleftrightarrow\quad \texttt{u[j, i]}.
\]

对采用 **C-order（行主序）连续内存** 的二维数组，\(N_y\) 行、\(N_x\) 列，零起始坐标 \((j,i)\) 对应的一维位置是

\[
\boxed{k=jN_x+i}.
\]

例如 \(N_x=6\)，位置 `u[2, 3]` 的一维下标为 \(k=2\times6+3=15\)。对于 `float64`（每个元素 8 字节），在连续内存中的相对字节偏移通常是 \(8k\)。

**注意**：这个公式只适合上述 C-order 的连续布局；对转置/步长切片视图，必须看 `strides`，不能盲目套用。

In [8]:
ny, nx = 3, 6
u = np.arange(ny * nx, dtype=np.float64).reshape(ny, nx)

j, i = 2, 3
k = j * nx + i
print("二维数组:\n", u)
print(f"u[{j}, {i}] =", u[j, i])
print(f"一维下标 k={k}, 展平后取值:", u.ravel(order="C")[k])

assert u[j, i] == u.ravel(order="C")[k]
print("索引映射验证通过 ✅")

二维数组:
 [[ 0.  1.  2.  3.  4.  5.]
 [ 6.  7.  8.  9. 10. 11.]
 [12. 13. 14. 15. 16. 17.]]
u[2, 3] = 15.0
一维下标 k=15, 展平后取值: 15.0
索引映射验证通过 ✅


### NumPy 数组的一个关键优势：整块数值计算

Python 列表的 `+` 是**拼接**，而 NumPy 数组的 `+` 是**逐元素加法**。NumPy 通常在底层实现数组运算，避免我们逐元素编写 Python 层循环。

但注意，**向量化不等于所有操作必然更快**：小数组、临时数组分配、不良内存访问模式都可能使收益下降。首先正确理解语义，然后再测量性能。

In [9]:
python_a = [1, 2, 3]
python_b = [10, 20, 30]
print("Python 列表相加:", python_a + python_b)

numpy_a = np.array([1, 2, 3])
numpy_b = np.array([10, 20, 30])
print("NumPy 数组相加:", numpy_a + numpy_b)
print("NumPy 乘法:", 2 * numpy_a)

Python 列表相加: [1, 2, 3, 10, 20, 30]
NumPy 数组相加: [11 22 33]
NumPy 乘法: [2 4 6]


## 6. 切片、视图和复制：真正影响数值程序正确性的细节

对于普通 Python `list`，`a[1:3]` 创建一个新的列表（浅拷贝）。但对于 NumPy：

```python
view = a[:, 1:3]
```

大多数**基本切片**返回**视图（view）**，与原数组共享数据。对视图的修改，通常会改变原始数组。相反，`a.copy()` 会创建独立数据。

在 CFD 里，这会影响边界条件、状态更新、缓存旧时刻数据等功能。

In [10]:
a = np.arange(12).reshape(3, 4)
view = a[:, 1:3]  # 基本切片：共享底层数据
copy = a[:, 1:3].copy()  # 明确复制

print("view 共享内存吗:", np.shares_memory(a, view))
print("copy 共享内存吗:", np.shares_memory(a, copy))

view[0, 0] = -99
print("修改 view 后的原数组:\n", a)

copy[0, 0] = 777
print("修改 copy 后的原数组不变:\n", a)

view 共享内存吗: True
copy 共享内存吗: False
修改 view 后的原数组:
 [[  0 -99   2   3]
 [  4   5   6   7]
 [  8   9  10  11]]
修改 copy 后的原数组不变:
 [[  0 -99   2   3]
 [  4   5   6   7]
 [  8   9  10  11]]


### 对 CFD 的具体影响：旧时间层和新时间层

错误写法：

```python
u_old = u
```

这只给同一个数组增加一个名字；修改 `u` 时 `u_old` 也会“变”。如果你希望保留上一个时间步的真实数值，应使用：

```python
u_old = u.copy()
```

另一个值得记住的规则：**NumPy 花式索引（如 `a[[0, 2]]`）通常返回副本；基本切片（如 `a[0:2]`）通常返回视图。**

In [11]:
u = np.array([1.0, 2.0, 3.0])
u_old_wrong = u
u_old_correct = u.copy()

u[0] = 100.0
print("更新后的 u:     ", u)
print("错误的 u_old:   ", u_old_wrong)
print("正确的 u_old:   ", u_old_correct)

b = np.arange(6)
fancy = b[[0, 2, 4]]
print("花式索引共享内存吗:", np.shares_memory(b, fancy))

更新后的 u:      [100.   2.   3.]
错误的 u_old:    [100.   2.   3.]
正确的 u_old:    [1. 2. 3.]
花式索引共享内存吗: False


## 7. 从数据结构到 CFD：二维 Laplace 算子

考虑在二维结构化网格上的标量场 \(\phi(x,y)\)。连续的 Laplace 算子是

\[
\nabla^2\phi=\frac{\partial^2\phi}{\partial x^2}+\frac{\partial^2\phi}{\partial y^2}.
\]

用中心差分得到常见的五点格式：

\[
(\nabla^2\phi)_{j,i}\approx
\frac{\phi_{j,i+1}-2\phi_{j,i}+\phi_{j,i-1}}{\Delta x^2}
+
\frac{\phi_{j+1,i}-2\phi_{j,i}+\phi_{j-1,i}}{\Delta y^2}.
\]

从**数据结构**角度看：

1. `phi` 存在一个形状为 `(ny, nx)` 的二维数组里。
2. 每个内部单元访问自身及其四邻居。
3. 各邻居的**相对索引偏移**固定，因此很适合结构化数组。
4. 这里只计算离散 Laplace 算子，**不是完整的 Poisson 求解器，更不是完整 CFD 求解器**。

NumPy 中，`phi[1:-1, 1:-1]` 表示所有内部单元（去除最外一层边界）。

In [12]:
ny, nx = 6, 8
dx = dy = 1.0
phi = np.zeros((ny, nx), dtype=np.float64)

# 创建一个简单的“高值区域”
phi[2:4, 3:5] = 1.0

lap = np.zeros_like(phi)
center = phi[1:-1, 1:-1]
right  = phi[1:-1, 2:]
left   = phi[1:-1, :-2]
up     = phi[2:, 1:-1]
down   = phi[:-2, 1:-1]

lap[1:-1, 1:-1] = ((right - 2 * center + left) / dx**2
                      + (up - 2 * center + down) / dy**2)

print("phi:\n", phi)
print("\n内部区域离散 Laplace 结果（边界行列暂设为 0）:\n", lap)
assert lap.shape == phi.shape

phi:
 [[0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 1. 0. 0. 0.]
 [0. 0. 0. 1. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]]

内部区域离散 Laplace 结果（边界行列暂设为 0）:
 [[ 0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  1.  1.  0.  0.  0.]
 [ 0.  0.  1. -2. -2.  1.  0.  0.]
 [ 0.  0.  1. -2. -2.  1.  0.  0.]
 [ 0.  0.  0.  1.  1.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.]]


### 为什么这里的“数据结构选择”会影响未来的程序架构？

若用 NumPy 二维数组，规则网格的左右上下邻居可由切片直接描述。对 **JAX** 也是类似的二维场表示思路，但 JAX 数组使用函数式的更新风格，不应依赖 NumPy 式的原地赋值。

如果将来处理**非结构网格**，某个单元的邻居不一定是 `i±1`、`j±1`。你可能需要**邻接表 / 图（第 06 讲）**；若进一步组装离散算子，经常会用到**稀疏矩阵（第 07 讲）**。

这就是从“会写语法”进阶为“知道如何表示一个问题”。

---
## 8. 动手练习（建议 20–30 分钟）

**先修改下面的 `TODO`，再运行检测单元。** 在你完成之前，检测单元会报告尚未通过，而不会让整个 Notebook 中断。最后才看参考答案。

### 练习 1：二维网格与一维索引互相转换

假设矩阵采用 C-order（行主序），有 \(N_x\) 列、\(N_y\) 行：

\[
k=jN_x+i,\qquad j=\lfloor k/N_x\rfloor,\qquad i=k\bmod N_x.
\]

补完两个函数，使 `(j, i)` 和 `k` 可以相互转换。**目标：两个转换都是 \(O(1)\)。**

In [13]:
def to_flat_index(j: int, i: int, nx: int) -> int:
    # TODO: 用 j, i, nx 计算 k
    return None


def from_flat_index(k: int, nx: int) -> tuple[int, int]:
    # TODO: 返回 (j, i)
    return None

### 练习 2：正确创建独立的二维列表

请创建一个 \(n_y\times n_x\) 的嵌套列表，所有元素初始值为 `fill`，并且不同的行**不共享同一内层列表**。

测试会修改第一行，验证第二行没有跟着变。

In [14]:
def make_python_grid(ny: int, nx: int, fill=0):
    # TODO: 返回互相独立的多行列表
    return None

### 练习 3：用 NumPy 实现五点 Laplace 算子

写函数 `laplacian_2d(phi, dx, dy)`：

- 参数 `phi` 为二维 NumPy 数组；
- 内部节点用本讲五点中心差分公式；
- 最外圈的输出暂时设为 0（这里只是教学约定，并不代表任何具体边界条件）；
- **不要修改输入的 `phi`**；
- 返回与 `phi` 形状相同的新数组。

提示：可以使用 `np.zeros_like(phi, dtype=np.float64)`；还可以借鉴第 7 节的切片方式。

In [15]:
def laplacian_2d(phi: np.ndarray, dx: float, dy: float) -> np.ndarray:
    # TODO: 这里实现二维五点差分；注意新建输出数组，不修改输入
    return None

### 一键检查（可以反复运行）

下列检测的失败并不会抛出异常中止整个 Notebook；它们帮助你判断还有哪些题尚未完成。

In [16]:
def check_exercise(name, tester):
    try:
        tester()
        print(f"✅ {name} 通过")
    except Exception as exc:
        print(f"⏳ {name} 未通过：{type(exc).__name__}: {exc}")


def test_ex1():
    assert to_flat_index(2, 3, 6) == 15
    assert from_flat_index(15, 6) == (2, 3)
    assert from_flat_index(to_flat_index(4, 1, 10), 10) == (4, 1)


def test_ex2():
    a = make_python_grid(3, 4, 0)
    assert a == [[0, 0, 0, 0] for _ in range(3)]
    a[0][1] = 99
    assert a[1][1] == 0


def test_ex3():
    a = np.zeros((5, 5), dtype=float)
    a[2, 2] = 1.0
    old = a.copy()
    result = laplacian_2d(a, 1.0, 1.0)
    assert isinstance(result, np.ndarray)
    assert result.shape == a.shape
    assert result[2, 2] == -4.0
    assert result[1, 2] == 1.0
    assert result[2, 1] == 1.0
    assert np.array_equal(a, old), "不应修改输入数组"
    # 非等距网格的检验：中心节点的 -2/dx²-2/dy²
    result2 = laplacian_2d(a, 0.5, 2.0)
    assert np.isclose(result2[2, 2], -2 / 0.5**2 - 2 / 2.0**2)

check_exercise("练习 1", test_ex1)
check_exercise("练习 2", test_ex2)
check_exercise("练习 3", test_ex3)

⏳ 练习 1 未通过：AssertionError: 
⏳ 练习 2 未通过：AssertionError: 
⏳ 练习 3 未通过：AssertionError: 


## 9. 参考答案（建议先独立尝试）

以下实现使用单独的 `_solution` 函数，不会覆盖你在练习区写的函数。如果想比较答案，可以直接运行这一节。

In [17]:
def to_flat_index_solution(j: int, i: int, nx: int) -> int:
    return j * nx + i


def from_flat_index_solution(k: int, nx: int) -> tuple[int, int]:
    return k // nx, k % nx


def make_python_grid_solution(ny: int, nx: int, fill=0):
    return [[fill for _ in range(nx)] for _ in range(ny)]


def laplacian_2d_solution(phi: np.ndarray, dx: float, dy: float) -> np.ndarray:
    out = np.zeros_like(phi, dtype=np.float64)
    out[1:-1, 1:-1] = (
        (phi[1:-1, 2:] - 2 * phi[1:-1, 1:-1] + phi[1:-1, :-2]) / dx**2
        + (phi[2:, 1:-1] - 2 * phi[1:-1, 1:-1] + phi[:-2, 1:-1]) / dy**2
    )
    return out

print("练习 1:", to_flat_index_solution(2, 3, 6), from_flat_index_solution(15, 6))
print("练习 2:", make_python_grid_solution(2, 3))
print("练习 3:\n", laplacian_2d_solution(np.eye(5), 1.0, 1.0))

练习 1: 15 (2, 3)
练习 2: [[0, 0, 0], [0, 0, 0]]
练习 3:
 [[ 0.  0.  0.  0.  0.]
 [ 0. -4.  2.  0.  0.]
 [ 0.  2. -4.  2.  0.]
 [ 0.  0.  2. -4.  0.]
 [ 0.  0.  0.  0.  0.]]


### 进阶挑战（选做）

1. **边界与正确性**：当 `phi` 只有 `(2, 2)` 大小时，五点差分的内部区域为空。你希望函数报错还是返回全零数组？请写出明确的使用约定。
2. **索引视角**：尝试用一维数组 `phi_flat` 保存场变量，使用 `k=j*nx+i` 实现同样的内部节点更新。比较它与二维数组切片在代码可读性上的差异。
3. **内存视角**：构造 `a = np.arange(12).reshape(3, 4)`，观察 `a.T.shape`、`a.T.strides`、`a.T.flags.c_contiguous`。解释转置为什么不一定需要复制数据。
4. **计算视角**：以 `float64` 保存一个 `1000×1000` 的场变量，理论上数据缓冲区需要约多少 MB？（提示：\(1000^2\times8\) 字节，使用十进制 MB。）

练习答案并不重要，重要的是能够解释**为什么这类表示方式是合理的**。

## 10. 本讲总结与自测

你应该能够不看材料回答：

1. **为什么** `list.append()` 均摊是 \(O(1)\)，而 `list.insert(0, x)` 是 \(O(n)\)？
2. `[[0] * nx] * ny` 为什么不适合创建可修改的二维网格？
3. Python `list` 与 NumPy `ndarray` 在数值存储方式上的主要区别是什么？
4. 对一个 C-order 的 `(ny, nx)` 数组，`(j, i)` 怎样映射到一维下标？
5. `view = u[:, 1:-1]` 和 `copy = u[:, 1:-1].copy()` 有何不同？
6. 为什么规则网格的五点模板适合二维数组，而非结构网格还需要图或邻接信息？

**下一讲：栈、队列与 `deque`。** 我们将手写最简单的栈和队列，比较时间复杂度，再实现一个小型广度优先搜索，作为后续图与非结构网格的铺垫。

**和你的 JAX CFD 学习的关系**：本讲最重要的不是背 API，而是理解 `shape / dtype / strides / view / copy` 与复杂度。等开始 JAX + GPU 计算时，这些概念会继续出现，只是可变性和底层执行模型有所不同。